# BioResearch: Overnight Campaign

Run autonomous ML research campaigns from Google Colab.

**Architecture:**
- **Colab** (this notebook): orchestration, Claude API calls, keep/revert decisions
- **Modal** (cloud GPUs): parallel experiment execution (5 seeds x H100/H200)

The orchestrator loops: ask Claude to modify `train.py` -> dispatch 5 seeds to Modal -> evaluate with statistical tests -> keep or revert -> repeat.

~100 experiments per overnight run.

## 1. Setup

In [ ]:
# Clone the repo (first time only)
!git clone https://github.com/mokalabs/bioresearch.git /content/bioresearch 2>/dev/null || echo 'Already cloned'
%cd /content/bioresearch

In [ ]:
# Install dependencies
!pip install -q numpy scipy pandas scikit-learn matplotlib anthropic modal

In [ ]:
import sys
sys.path.insert(0, '/content/bioresearch')

## 2. Authentication

You need two keys:
1. **Anthropic API key** — for Claude to propose modifications
2. **Modal token** — for dispatching experiments to cloud GPUs

Set these in Colab Secrets (key icon in sidebar) or paste below.

In [ ]:
import os

# Option A: Use Colab Secrets (recommended — click the key icon in the left sidebar)
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    os.environ['MODAL_TOKEN_ID'] = userdata.get('MODAL_TOKEN_ID')
    os.environ['MODAL_TOKEN_SECRET'] = userdata.get('MODAL_TOKEN_SECRET')
    print('Loaded keys from Colab Secrets')
except Exception:
    pass

# Option B: Paste directly (less secure but works)
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
# os.environ['MODAL_TOKEN_ID'] = '...'
# os.environ['MODAL_TOKEN_SECRET'] = '...'

# Verify
assert 'ANTHROPIC_API_KEY' in os.environ, 'Set ANTHROPIC_API_KEY'
print(f'Anthropic key: ...{os.environ["ANTHROPIC_API_KEY"][-6:]}')
print(f'Modal token: {"set" if "MODAL_TOKEN_ID" in os.environ else "NOT SET"}')

In [ ]:
# Verify Modal connection
from infra.colab import colab_setup
status = colab_setup()

## 3. Choose Your Domain

| Domain | What it does | Baseline | Primary Metric |
|---|---|---|---|
| `perturbation` | Predict gene expression after perturbation | Linear model | `pearson_deg` |
| `molecules` | 22-endpoint ADMET profiling + molecular generation | Ridge/LogReg | `composite_admet` |
| `trials` | Clinical trial success prediction (chains all three) | GradientBoosting | `auroc` |

In [ ]:
# === CONFIGURE YOUR CAMPAIGN HERE ===

DOMAIN = 'perturbation'   # 'perturbation', 'molecules', or 'trials'
ITERATIONS = 100           # number of agent iterations (~5-10 min each with Modal)
NUM_SEEDS = 5              # seeds per experiment (statistical rigor)
TIME_BUDGET = 600          # seconds per seed
USE_MODAL = True           # True = Modal cloud GPUs, False = run locally in Colab
POPULATION = 0             # 0 = single agent, 2-4 = population search with tournament

## 4. Run the Campaign

This cell runs the full autoresearch loop. It will:
1. Run baseline experiments (5 seeds)
2. Ask Claude to propose a modification
3. Run the modified code across 5 seeds (on Modal or locally)
4. Evaluate: Welch's t-test (p < 0.05) + Cohen's d (> 0.3) + guard rails
5. Keep or revert
6. Repeat

**Colab will stay alive** as long as this cell is running. For overnight runs:
- Keep the browser tab open (Colab disconnects after ~90min of inactivity)
- Use a Colab Pro/Pro+ subscription for longer runtime limits
- The [Colab Keep Alive](https://chrome.google.com/webstore/detail/colab-alive/eihnaflcafllhmdojdknpfcjnbighfkb) browser extension can help prevent disconnects

In [ ]:
from pathlib import Path
from engine.loop import LoopConfig, autoresearch_loop
from engine.orchestrator import OrchestratorConfig
from engine.metrics import MetricSpec, MetricRole, MetricDirection

PROJECT_ROOT = Path('/content/bioresearch')

# --- Metric specs per domain ---
METRIC_SPECS = {
    'perturbation': [
        MetricSpec('pearson_deg', MetricRole.PRIMARY, MetricDirection.HIGHER),
        MetricSpec('mse_top20_deg', MetricRole.GUARD, MetricDirection.LOWER, guard_threshold=0.1),
        MetricSpec('direction_acc', MetricRole.GUARD, MetricDirection.HIGHER, guard_threshold=0.3),
        MetricSpec('cross_context', MetricRole.BONUS, MetricDirection.LOWER),
        MetricSpec('pearson_all', MetricRole.DIAGNOSTIC, MetricDirection.HIGHER),
    ],
    'molecules': [
        MetricSpec('composite_admet', MetricRole.PRIMARY, MetricDirection.HIGHER),
        MetricSpec('gen_profile_match', MetricRole.PRIMARY, MetricDirection.HIGHER),
        MetricSpec('gen_diversity', MetricRole.GUARD, MetricDirection.HIGHER, guard_threshold=0.3),
    ],
    'trials': [
        MetricSpec('auroc', MetricRole.PRIMARY, MetricDirection.HIGHER),
        MetricSpec('calibration_ece', MetricRole.GUARD, MetricDirection.LOWER, guard_threshold=0.15),
        MetricSpec('net_value', MetricRole.GUARD, MetricDirection.HIGHER, guard_threshold=0.0),
        MetricSpec('lift_at_10', MetricRole.BONUS, MetricDirection.HIGHER),
    ],
}

metric_specs = METRIC_SPECS[DOMAIN]
domain_dir = str(PROJECT_ROOT / 'domains' / DOMAIN)
output_dir = str(PROJECT_ROOT / 'results' / DOMAIN)

# --- Set up Modal runner ---
run_seeds_parallel = None
if USE_MODAL:
    from cli import _make_modal_runner
    run_seeds_parallel = _make_modal_runner(DOMAIN)
    if run_seeds_parallel:
        print('Modal runner ready: experiments will dispatch to cloud GPUs')
    else:
        print('Modal setup failed, falling back to local execution')

# --- Set up knowledge function ---
from cli import _make_knowledge_fn
knowledge_fn = _make_knowledge_fn(DOMAIN)

# --- Orchestrator config ---
orch_config = OrchestratorConfig(model='claude-opus-4-6')

print(f'\nCampaign: {DOMAIN}')
print(f'Iterations: {ITERATIONS}')
print(f'Seeds per experiment: {NUM_SEEDS}')
print(f'Time budget per seed: {TIME_BUDGET}s')
print(f'Execution: {"Modal (cloud GPU)" if run_seeds_parallel else "Local"}')
print(f'Population: {POPULATION if POPULATION > 1 else "single agent"}')

In [ ]:
# === RUN THE CAMPAIGN ===
# This is the main cell. It runs until all iterations complete or you interrupt it.
# Progress is printed live. Results are saved to results/<domain>/.

if POPULATION > 1:
    from engine.population import PopulationSearch, PopulationConfig
    pop_config = PopulationConfig(
        num_agents=POPULATION,
        domain_dir=domain_dir,
        output_dir=output_dir,
        num_seeds=NUM_SEEDS,
        max_iterations=ITERATIONS,
        time_budget_per_seed=TIME_BUDGET,
        orchestrator_config=orch_config,
    )
    search = PopulationSearch(
        metric_specs=metric_specs,
        config=pop_config,
        knowledge_fn=knowledge_fn,
        run_seeds_parallel=run_seeds_parallel,
    )
    search.run()
else:
    config = LoopConfig(
        domain_dir=domain_dir,
        output_dir=output_dir,
        num_seeds=NUM_SEEDS,
        max_iterations=ITERATIONS,
        time_budget_per_seed=TIME_BUDGET,
        orchestrator_config=orch_config,
    )
    tracker = autoresearch_loop(
        metric_specs=metric_specs,
        loop_config=config,
        knowledge_fn=knowledge_fn,
        run_seeds_parallel=run_seeds_parallel,
    )

## 5. Review Results

In [ ]:
from engine.tracker import ExperimentTracker
from IPython.display import display, Image, Markdown
import json

tracker = ExperimentTracker(output_dir)
tracker.load_existing()

# Campaign summary
summary = tracker.generate_summary()
display(Markdown(f'```\n{summary}\n```'))

# Best result
for spec in metric_specs:
    if spec.role == MetricRole.PRIMARY:
        best = tracker.get_best_record(spec.name)
        if best:
            print(f'\nBest {spec.name}: {best.metrics.get(spec.name, 0):.6f}')
            print(f'  Iteration: {best.iteration}')
            print(f'  Description: {best.description}')

In [ ]:
# Plot metric progression
from pathlib import Path

metric_names = [s.name for s in metric_specs]
directions = {s.name: s.direction.value for s in metric_specs}
tracker.plot_all_metrics(metric_names, directions)
tracker.plot_experiment_overview()

# Display plots (saved to plots/ subdir)
plots_dir = Path(output_dir) / 'plots'
for png in sorted(plots_dir.glob('*.png')):
    print(f'\n{png.name}')
    display(Image(filename=str(png)))

In [ ]:
# View the best train.py that the agent discovered
best_train = (Path(domain_dir) / 'train.py').read_text()
print(best_train)

In [ ]:
# Download results to your local machine
try:
    from google.colab import files
    import shutil
    shutil.make_archive('/content/bioresearch_results', 'zip', output_dir)
    files.download('/content/bioresearch_results.zip')
except ImportError:
    print(f'Results saved to: {output_dir}')